# Unified Ultimatum Game Steering Analysis
## LLM-vs-LLM, LLM-vs-RuleBased, and LLM-vs-LLM (Clean): A Comprehensive Statistical Report

This notebook provides a full empirical analysis of activation-steered ultimatum game experiments.
We compare **three** evaluation paradigms:

- **LLM-vs-LLM**: both proposer and responder are steered or unsteered Qwen 7B agents (teammate's data)
- **LLM-vs-RuleBased** *(7B only)*: proposer or responder is a Qwen 7B LLM; the counterpart follows a fixed rule-based threshold policy (teammate's data)
- **LLM-vs-LLM (Clean)**: our paired-design experiments with variable pools, temp=0, general-domain steering vectors on Qwen 7B. Full factorial grid: 10 dims x 9 layers (L4-L20) x 5 alphas (-7, -5, 5, 7, 15) = 450 configs.

**Structure:** Data loading -> per-experiment summaries -> statistical tests -> effect decomposition -> top-config leaderboards -> cross-paradigm comparison

> Payoff is a *compound* outcome: `E[payoff] = accept_rate * mean_offer_pct`. Steering can simultaneously shift both components in opposing directions, so we decompose effects throughout.

---
## 0 - Setup

In [ ]:
import json, glob, os, warnings, re
from pathlib import Path
from itertools import product

import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import Normalize, TwoSlopeNorm
from matplotlib.cm import ScalarMappable
import seaborn as sns

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:.3f}'.format)

# -- publication-quality style ---
plt.rcParams.update({
    'font.family':       'serif',
    'font.size':         11,
    'axes.titlesize':    12,
    'axes.labelsize':    11,
    'legend.fontsize':   9,
    'xtick.labelsize':   9,
    'ytick.labelsize':   9,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'figure.dpi':        120,
    'savefig.dpi':       200,
    'axes.grid':         True,
    'grid.alpha':        0.35,
    'grid.linestyle':    '--',
})

PALETTE = sns.color_palette('tab10')
DIVERGING = sns.color_palette('RdBu_r', 12)

# -- Paths: try local first, then teammate's UCL lab path ---
# Detect project root by looking for ultimatum_game.py
_candidates = [Path('.').resolve(), Path('..').resolve(), Path('../..').resolve()]
PROJECT_ROOT = _candidates[0]
for _c in _candidates:
    if (_c / 'ultimatum_game.py').exists():
        PROJECT_ROOT = _c
        break

TEAMMATE_ROOT = Path('/cs/student/projects3/2022/dsurrao/comp0087_snlp_cwk/results/ultimatum')
LOCAL_ROOT = PROJECT_ROOT / 'results' / 'ultimatum'

# Use teammate path if available, else fall back to local
if TEAMMATE_ROOT.exists():
    ROOT = TEAMMATE_ROOT
else:
    ROOT = LOCAL_ROOT
    print(f'NOTE: Teammate root {TEAMMATE_ROOT} not available, using local {ROOT}')

LVL_DIR  = ROOT / 'llm_vs_llm'
LVRB_DIR = ROOT / 'llm_vs_rulebased'

# Clean pipeline data: single directory with the final 7B LLM-vs-LLM grid (450 files)
CLEAN_DIR = LOCAL_ROOT / 'final_7b_llm_vs_llm'

print(f'Project root:      {PROJECT_ROOT}')
print(f'Teammate root:     {ROOT} (exists: {ROOT.exists()})')
print(f'Clean pipeline:    {CLEAN_DIR} (exists: {CLEAN_DIR.exists()})')
if LVL_DIR.exists():
    print(f'LvL experiments:   {[p.name for p in LVL_DIR.iterdir() if p.is_dir()]}')
if LVRB_DIR.exists():
    print(f'LvRB 7B experiments: {[p.name for p in LVRB_DIR.iterdir() if p.is_dir() and "7b" in p.name.lower()]}')

# Report clean pipeline file count
n_clean = len(list(CLEAN_DIR.glob('*.json'))) if CLEAN_DIR.exists() else 0
print(f'  final_7b_llm_vs_llm    : {n_clean:3d} files {"" if CLEAN_DIR.exists() else "(missing)"}')

---
## 1 - Data Loading

In [ ]:
def _safe_sub(a, b):
    """Subtract two values, returning NaN if either is None/NaN."""
    if a is None or b is None:
        return np.nan
    try:
        return float(a) - float(b)
    except (TypeError, ValueError):
        return np.nan

def _safe_mul(a, b):
    """Multiply two values, returning NaN if either is None/NaN."""
    if a is None or b is None:
        return np.nan
    try:
        return float(a) * float(b)
    except (TypeError, ValueError):
        return np.nan

def load_final_best(path: Path) -> 'dict | None':
    """Load a final_best.json (teammate format) and flatten it into a record dict."""
    try:
        d = json.loads(path.read_text())
    except Exception:
        return None

    parts = path.parts
    dim   = parts[-2]
    role  = parts[-3]
    layer = parts[-4]
    exp   = parts[-5]

    bl  = d.get('baseline_summary', {})
    fin = d.get('final_summary',    {})
    fs  = d.get('final_stats',      {})

    offer_shift   = fs.get('offer_shift',  {})
    accept_shift  = fs.get('acceptance_shift', {})
    payoff_ref    = fs.get('payoff_delta_reference_only', {})

    scores = d.get('scores_per_alpha', {})

    rec = dict(
        experiment   = exp,
        layer        = layer,
        layer_num    = int(layer.lstrip('Ll')),
        role         = role,
        dimension    = dim,
        model        = d.get('model', 'unknown'),
        best_alpha   = d.get('best_alpha'),
        best_fitness = d.get('best_fitness'),
        n_games      = bl.get('n_games', 0),
        bl_accept_rate       = bl.get('accept_rate'),
        bl_mean_proposer_pct = bl.get('mean_proposer_pct'),
        bl_mean_proposer_payoff = bl.get('mean_proposer_payoff_pct'),
        bl_mean_responder_payoff = bl.get('mean_responder_payoff_pct'),
        bl_n_games           = bl.get('n_games'),
        bl_n_valid           = bl.get('n_valid'),
        bl_n_parse_errors    = bl.get('n_parse_errors', 0),
        fin_accept_rate       = fin.get('accept_rate'),
        fin_mean_proposer_pct = fin.get('mean_proposer_pct'),
        fin_mean_proposer_payoff = fin.get('mean_proposer_payoff_pct'),
        fin_mean_responder_payoff = fin.get('mean_responder_payoff_pct'),
        fin_n_valid          = fin.get('n_valid'),
        fin_n_parse_errors   = fin.get('n_parse_errors', 0),
        delta_accept_rate       = (fin.get('accept_rate', np.nan) or np.nan) - (bl.get('accept_rate', np.nan) or np.nan),
        delta_proposer_pct      = (fin.get('mean_proposer_pct', np.nan) or np.nan) - (bl.get('mean_proposer_pct', np.nan) or np.nan),
        delta_proposer_payoff   = (fin.get('mean_proposer_payoff_pct', np.nan) or np.nan) - (bl.get('mean_proposer_payoff_pct', np.nan) or np.nan),
        delta_responder_payoff  = (fin.get('mean_responder_payoff_pct', np.nan) or np.nan) - (bl.get('mean_responder_payoff_pct', np.nan) or np.nan),
        offer_p          = offer_shift.get('p_value'),
        offer_cohens_d   = offer_shift.get('cohens_d'),
        offer_sig        = offer_shift.get('significant_at_0_05', False),
        accept_p         = accept_shift.get('p_value'),
        accept_sig       = accept_shift.get('significant_at_0_05', False),
        payoff_p         = payoff_ref.get('p_value'),
        payoff_cohens_d  = payoff_ref.get('cohens_d'),
        scores_per_alpha = scores,
    )
    rec['bl_compound_payoff']  = (rec['bl_accept_rate']  or np.nan) * (rec['bl_mean_proposer_pct']  or np.nan) / 100
    rec['fin_compound_payoff'] = (rec['fin_accept_rate'] or np.nan) * (rec['fin_mean_proposer_pct'] or np.nan) / 100
    rec['delta_compound_payoff'] = rec['fin_compound_payoff'] - rec['bl_compound_payoff']
    return rec


def load_games(path: Path) -> 'pd.DataFrame | None':
    """Load a games_*.json (teammate format) and return a tidy DataFrame."""
    try:
        d = json.loads(path.read_text())
    except Exception:
        return None
    games = d.get('games', [])
    if not games:
        return None
    df = pd.DataFrame(games)
    parts = path.parts
    df['dimension'] = parts[-2]
    df['role']      = parts[-3]
    df['layer']     = parts[-4]
    df['experiment']= parts[-5]
    df['alpha_tag'] = d.get('tag', path.stem.replace('games_', ''))
    df['alpha_val'] = d.get('alpha', np.nan)
    df['proposer_pct'] = df['proposer_share'] / df['pool'] * 100
    df['responder_pct_actual'] = df['responder_share'] / df['pool'] * 100
    df['proposer_payoff_pct']  = df['proposer_payoff'] / df['pool'] * 100
    df['responder_payoff_pct'] = df['responder_payoff'] / df['pool'] * 100
    return df


def load_clean_pipeline_file(path: Path) -> 'dict | None':
    """Load a single clean-pipeline JSON and convert to the same record schema."""
    try:
        d = json.loads(path.read_text())
    except Exception:
        return None

    cfg = d.get('config', {})
    s   = d.get('summary', {})
    if not cfg or not s:
        return None

    dimension = cfg.get('dimension', 'unknown')
    layers    = cfg.get('layers', [0])
    layer_num = layers[0] if layers else 0
    alpha     = cfg.get('alpha', 0)
    n_games   = s.get('n_games', cfg.get('n_games', 0))
    role      = cfg.get('steered_role', 'proposer')
    source_dir = path.parent.name

    bl_accept  = s.get('baseline_accept_rate', np.nan)
    fin_accept = s.get('steered_accept_rate', np.nan)
    bl_offer   = s.get('baseline_mean_proposer_pct', np.nan)
    fin_offer  = s.get('steered_mean_proposer_pct', np.nan)

    rec = dict(
        experiment   = source_dir,
        layer        = f'L{layer_num}',
        layer_num    = layer_num,
        role         = role,
        dimension    = dimension,
        model        = cfg.get('model', 'qwen2.5-7b'),
        best_alpha   = alpha,
        best_fitness = None,
        n_games      = n_games,
        bl_accept_rate       = bl_accept,
        bl_mean_proposer_pct = bl_offer,
        bl_mean_proposer_payoff = s.get('baseline_mean_payoff_pct'),
        bl_mean_responder_payoff = None,
        bl_n_games           = n_games,
        bl_n_valid           = s.get('n_valid', n_games),
        bl_n_parse_errors    = s.get('n_parse_errors', 0),
        fin_accept_rate       = fin_accept,
        fin_mean_proposer_pct = fin_offer,
        fin_mean_proposer_payoff = s.get('steered_mean_payoff_pct'),
        fin_mean_responder_payoff = None,
        fin_n_valid          = s.get('n_valid', n_games),
        fin_n_parse_errors   = s.get('n_parse_errors', 0),
        delta_accept_rate       = _safe_sub(fin_accept, bl_accept),
        delta_proposer_pct      = _safe_sub(fin_offer, bl_offer),
        delta_proposer_payoff   = _safe_sub(s.get('steered_mean_payoff_pct'), s.get('baseline_mean_payoff_pct')),
        delta_responder_payoff  = None,
        offer_p          = s.get('paired_ttest_p'),
        offer_cohens_d   = s.get('cohens_d'),
        offer_sig        = (s.get('paired_ttest_p') or 1.0) < 0.05,
        accept_p         = None,
        accept_sig       = False,
        payoff_p         = None,
        payoff_cohens_d  = None,
        scores_per_alpha = {},
    )
    rec['bl_compound_payoff']  = _safe_mul(bl_accept, bl_offer) / 100
    rec['fin_compound_payoff'] = _safe_mul(fin_accept, fin_offer) / 100
    rec['delta_compound_payoff'] = rec['fin_compound_payoff'] - rec['bl_compound_payoff']
    return rec


def load_clean_pipeline_games(path: Path) -> 'pd.DataFrame | None':
    """Load games from a clean-pipeline JSON and convert to tidy format."""
    try:
        d = json.loads(path.read_text())
    except Exception:
        return None

    cfg   = d.get('config', {})
    games = d.get('games', [])
    if not games:
        return None

    dimension = cfg.get('dimension', 'unknown')
    layers    = cfg.get('layers', [0])
    layer_num = layers[0] if layers else 0
    alpha     = cfg.get('alpha', 0)
    role      = cfg.get('steered_role', 'proposer')
    source_dir = path.parent.name

    rows = []
    for g in games:
        steered  = g.get('steered', {})
        baseline = g.get('baseline', {})
        pool = g.get('pool', steered.get('pool', baseline.get('pool', 100)))

        # Steered game row
        rows.append({
            'game_id':           g.get('game_id'),
            'pool':              pool,
            'proposer_share':    steered.get('proposer_share', np.nan),
            'responder_share':   steered.get('responder_share', np.nan),
            'agreed':            steered.get('agreed', False),
            'proposer_payoff':   steered.get('proposer_payoff', 0),
            'responder_payoff':  steered.get('responder_payoff', 0),
            'parse_error':       steered.get('parse_error') is not None,
            'dimension':         dimension,
            'role':              role,
            'layer':             f'L{layer_num}',
            'experiment':        source_dir,
            'alpha_tag':         f'alpha{alpha}',
            'alpha_val':         float(alpha),
        })
        # Baseline game row
        rows.append({
            'game_id':           g.get('game_id'),
            'pool':              pool,
            'proposer_share':    baseline.get('proposer_share', np.nan),
            'responder_share':   baseline.get('responder_share', np.nan),
            'agreed':            baseline.get('agreed', False),
            'proposer_payoff':   baseline.get('proposer_payoff', 0),
            'responder_payoff':  baseline.get('responder_payoff', 0),
            'parse_error':       baseline.get('parse_error') is not None,
            'dimension':         dimension,
            'role':              role,
            'layer':             f'L{layer_num}',
            'experiment':        source_dir,
            'alpha_tag':         'baseline',
            'alpha_val':         0.0,
        })

    df = pd.DataFrame(rows)
    df['proposer_pct']         = df['proposer_share'] / df['pool'] * 100
    df['responder_pct_actual'] = df['responder_share'] / df['pool'] * 100
    df['proposer_payoff_pct']  = df['proposer_payoff'] / df['pool'] * 100
    df['responder_payoff_pct'] = df['responder_payoff'] / df['pool'] * 100
    return df


print('Loader functions defined (teammate + clean-pipeline).')

In [ ]:
# == LLM vs LLM: all experiments ==
lvl_records = []
lvl_games   = []

if LVL_DIR.exists():
    for fb_path in sorted(LVL_DIR.rglob('final_best.json')):
        rec = load_final_best(fb_path)
        if rec:
            rec['paradigm'] = 'LLM-vs-LLM'
            lvl_records.append(rec)

    for g_path in sorted(LVL_DIR.rglob('games_*.json')):
        gdf = load_games(g_path)
        if gdf is not None:
            gdf['paradigm'] = 'LLM-vs-LLM'
            lvl_games.append(gdf)
else:
    print('LLM-vs-LLM directory not found -- skipping.')

# == LLM vs Rule-Based: 7B only ==
lvrb_records = []
lvrb_games   = []

if LVRB_DIR.exists():
    for exp_dir in LVRB_DIR.iterdir():
        if not exp_dir.is_dir() or '7b' not in exp_dir.name.lower():
            continue
        for fb_path in sorted(exp_dir.rglob('final_best.json')):
            rec = load_final_best(fb_path)
            if rec:
                rec['paradigm'] = 'LLM-vs-RuleBased'
                lvrb_records.append(rec)
        for g_path in sorted(exp_dir.rglob('games_*.json')):
            gdf = load_games(g_path)
            if gdf is not None:
                gdf['paradigm'] = 'LLM-vs-RuleBased'
                lvrb_games.append(gdf)
else:
    print('LLM-vs-RuleBased directory not found -- skipping.')

print(f'Teammate records: LvL={len(lvl_records)}, LvRB={len(lvrb_records)}')

In [ ]:
# == LLM-vs-LLM (Clean): our paired-design experiments (final grid) ==
# All 450 configs live in one directory -- no dedup needed.
clean_files = sorted(CLEAN_DIR.glob('*.json')) if CLEAN_DIR.exists() else []

clean_records = []
all_clean_games = []

for json_path in clean_files:
    rec = load_clean_pipeline_file(json_path)
    if rec is None:
        continue
    rec['paradigm'] = 'LLM-vs-LLM (Clean)'
    clean_records.append(rec)

    gdf = load_clean_pipeline_games(json_path)
    if gdf is not None:
        gdf['paradigm'] = 'LLM-vs-LLM (Clean)'
        all_clean_games.append(gdf)

print(f'LLM-vs-LLM (Clean): {len(clean_records)} configs loaded from {CLEAN_DIR.name}/')
print(f'  Dimensions: {sorted(set(r["dimension"] for r in clean_records))}')
print(f'  Layers:     {sorted(set(r["layer_num"] for r in clean_records))}')
print(f'  Alphas:     {sorted(set(r["best_alpha"] for r in clean_records))}')
print(f'  Roles:      {sorted(set(r["role"] for r in clean_records))}')
print(f'  Game DFs:   {len(all_clean_games)}')

In [ ]:
# == Merge all paradigms ==
all_records = lvl_records + lvrb_records + clean_records
all_games_list = lvl_games + lvrb_games + all_clean_games

df_sum  = pd.DataFrame(all_records)
if all_games_list:
    df_games = pd.concat(all_games_list, ignore_index=True)
else:
    df_games = pd.DataFrame()

# drop scores_per_alpha (nested dict, not needed in main table)
if 'scores_per_alpha' in df_sum.columns:
    scores_col = df_sum.pop('scores_per_alpha')

print(f'Summary records  : {len(df_sum):,}  ({df_sum["paradigm"].value_counts().to_dict()})')
print(f'Game records     : {len(df_games):,}')
print(f'Dimensions       : {sorted(df_sum["dimension"].unique())}')
print(f'Layers           : {sorted(df_sum["layer_num"].unique())}')
print(f'Roles            : {sorted(df_sum["role"].unique())}')
print(f'Paradigms        : {sorted(df_sum["paradigm"].unique())}')

---
## 2 - Dataset Overview

In [ ]:
cov = (
    df_sum.groupby(['paradigm', 'role', 'layer_num'])
    .agg(n_configs=('dimension', 'count'),
         dimensions=('dimension', lambda x: ', '.join(sorted(x.unique()))))
    .reset_index()
)
print('=== Configuration Coverage ===')
display(cov)

In [ ]:
paradigm_colors = {p: PALETTE[i] for i, p in enumerate(sorted(df_sum['paradigm'].unique()))}

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# (a) configs per paradigm
cnt = df_sum['paradigm'].value_counts()
colors_a = [paradigm_colors.get(p, 'grey') for p in cnt.index]
axes[0].bar(cnt.index, cnt.values, color=colors_a, edgecolor='k', linewidth=0.6)
axes[0].set_title('(a) Configs per Paradigm')
axes[0].set_ylabel('# Configurations')
axes[0].tick_params(axis='x', rotation=15)
for bar, v in zip(axes[0].patches, cnt.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, str(v),
                 ha='center', va='bottom', fontsize=9)

# (b) configs per dimension x paradigm
piv = df_sum.groupby(['paradigm', 'dimension']).size().unstack(fill_value=0)
piv_colors = [paradigm_colors.get(p, 'grey') for p in piv.index]
piv.T.plot(kind='bar', ax=axes[1], color=piv_colors, edgecolor='k',
           linewidth=0.5, width=0.7)
axes[1].set_title('(b) Configs per Dimension')
axes[1].set_xlabel('Dimension')
axes[1].set_ylabel('# Configs')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend(fontsize=7)

# (c) configs per layer x paradigm
piv2 = df_sum.groupby(['paradigm', 'layer_num']).size().unstack(fill_value=0)
piv2_colors = [paradigm_colors.get(p, 'grey') for p in piv2.index]
piv2.T.plot(kind='bar', ax=axes[2], color=piv2_colors, edgecolor='k',
            linewidth=0.5, width=0.7)
axes[2].set_title('(c) Configs per Layer')
axes[2].set_xlabel('Layer')
axes[2].set_ylabel('# Configs')
axes[2].legend(fontsize=7)

plt.suptitle('Dataset Coverage Overview (3 Paradigms)', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 3 - Baseline Behaviour

Before examining steering effects we characterise the unsteered (baseline) offer and acceptance distributions.

In [ ]:
bl_games = pd.DataFrame()
if not df_games.empty and 'alpha_tag' in df_games.columns:
    bl_games = df_games[df_games['alpha_tag'] == 'baseline'].copy()

if not bl_games.empty:
    paradigm_list = sorted(bl_games['paradigm'].unique())
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    for i, par in enumerate(paradigm_list):
        grp = bl_games[bl_games['paradigm'] == par]
        axes[0].hist(grp['proposer_pct'].dropna(), bins=30, alpha=0.5, label=par,
                     color=PALETTE[i], edgecolor='k', linewidth=0.3)
    axes[0].axvline(50, color='red', lw=1.2, linestyle='--', label='50% (equal split)')
    axes[0].set_title('(a) Baseline: Proposer Offer (%)')
    axes[0].set_xlabel('Proposer share of pool (%)')
    axes[0].set_ylabel('Count')
    axes[0].legend(fontsize=7)

    bl_acc = bl_games.groupby(['paradigm', 'dimension'])['agreed'].mean().unstack('paradigm') * 100
    bl_acc.plot(kind='bar', ax=axes[1], edgecolor='k', linewidth=0.5, width=0.7)
    axes[1].set_title('(b) Baseline Accept Rate by Dimension')
    axes[1].set_xlabel('Dimension')
    axes[1].set_ylabel('Acceptance rate (%)')
    axes[1].set_ylim(0, 110)
    axes[1].tick_params(axis='x', rotation=45)
    axes[1].legend(fontsize=7)

    for i, par in enumerate(paradigm_list):
        grp = bl_games[bl_games['paradigm'] == par]
        axes[2].hist(grp['pool'].dropna(), bins=25, alpha=0.5, label=par,
                     color=PALETTE[i], edgecolor='k', linewidth=0.3)
    axes[2].set_title('(c) Pool Size Distribution')
    axes[2].set_xlabel('Pool size ($)')
    axes[2].set_ylabel('Count')
    axes[2].legend(fontsize=7)

    plt.suptitle('Baseline Game Characteristics (All Paradigms)', fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

    bl_summary = bl_games.groupby('paradigm').agg(
        n_games=('agreed', 'count'),
        accept_rate=('agreed', 'mean'),
        mean_offer_pct=('proposer_pct', 'mean'),
        std_offer_pct=('proposer_pct', 'std'),
        mean_pool=('pool', 'mean'),
    ).round(3)
    print('\nBaseline Summary')
    display(bl_summary)
else:
    print('No baseline games available.')

---
## 4 - Payoff Decomposition: The Compound Variable

Proposer expected payoff is:
$$\mathbb{E}[\text{payoff}] = \underbrace{\text{accept\_rate}}_{\text{acceptance component}} \times \underbrace{\text{offer\_pct}}_{\text{extraction component}}$$

Steering can move these in opposite directions. We therefore track all three quantities independently.

In [ ]:
if not df_games.empty:
    df_games['compound_payoff_pct'] = df_games['agreed'].astype(float) * df_games['proposer_pct']

    # Pair steered games with their baseline counterparts
    bl_g = df_games[df_games['alpha_tag'] == 'baseline'][
        ['paradigm','experiment','layer','dimension','role','game_id',
         'proposer_pct','agreed','compound_payoff_pct']].copy()

    # For teammate data: use 'final' tag. For clean pipeline: use the steered alpha tag
    has_final = 'final' in df_games['alpha_tag'].values
    fin_mask = df_games['alpha_tag'] == 'final' if has_final else (df_games['alpha_tag'] != 'baseline')
    fin_g = df_games[fin_mask][
        ['paradigm','experiment','layer','dimension','role','game_id',
         'proposer_pct','agreed','compound_payoff_pct']].copy()

    bl_g  = bl_g.rename(columns={c: 'bl_'+c  for c in ['proposer_pct','agreed','compound_payoff_pct']})
    fin_g = fin_g.rename(columns={c: 'fin_'+c for c in ['proposer_pct','agreed','compound_payoff_pct']})

    paired = pd.merge(bl_g, fin_g,
                      on=['paradigm','experiment','layer','dimension','role','game_id'],
                      how='inner')
    paired['delta_offer']    = paired['fin_proposer_pct'] - paired['bl_proposer_pct']
    paired['delta_agreed']   = paired['fin_agreed'].astype(float) - paired['bl_agreed'].astype(float)
    paired['delta_compound'] = paired['fin_compound_payoff_pct'] - paired['bl_compound_payoff_pct']

    print(f'Paired game records: {len(paired):,}')

    agg_paired = paired.groupby(['paradigm','experiment','layer','dimension','role']).agg(
        delta_offer=('delta_offer','mean'),
        delta_accept=('delta_agreed','mean'),
        delta_compound=('delta_compound','mean'),
    ).reset_index()

    paradigm_list = sorted(agg_paired['paradigm'].unique())
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for i, par in enumerate(paradigm_list):
        grp = agg_paired[agg_paired['paradigm'] == par]
        axes[0].scatter(grp['delta_offer'], grp['delta_accept']*100,
                        c=[PALETTE[i]]*len(grp), label=par, alpha=0.6, s=40,
                        edgecolors='k', linewidths=0.3)

    axes[0].axhline(0, color='k', lw=0.8, linestyle='--')
    axes[0].axvline(0, color='k', lw=0.8, linestyle='--')
    axes[0].set_xlabel('Delta Proposer offer (%)')
    axes[0].set_ylabel('Delta Accept rate (pp)')
    axes[0].set_title('(a) Offer vs Acceptance Trade-off')
    axes[0].legend(fontsize=7)

    for i, par in enumerate(paradigm_list):
        grp = agg_paired[agg_paired['paradigm'] == par]
        axes[1].hist(grp['delta_compound'], bins=25, alpha=0.5, label=par,
                     color=PALETTE[i], edgecolor='k', linewidth=0.3)
    axes[1].axvline(0, color='red', lw=1.2, linestyle='--', label='no change')
    axes[1].set_xlabel('Delta Compound payoff (pp)')
    axes[1].set_ylabel('# Configs')
    axes[1].set_title('(b) Distribution of Compound Payoff Change')
    axes[1].legend(fontsize=7)

    plt.suptitle('Payoff Decomposition: Offer x Acceptance (All Paradigms)', fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print('No game-level data available for payoff decomposition.')

---
## 5 - Statistical Significance and Multiple Testing Correction

In [ ]:
df_stats = df_sum.copy()

for col in ['offer_p','accept_p','payoff_p','offer_cohens_d','payoff_cohens_d']:
    if col in df_stats.columns:
        df_stats[col] = pd.to_numeric(df_stats[col], errors='coerce')

# BH-FDR correction on offer p-values
if 'offer_p' in df_stats.columns:
    valid_offer = df_stats['offer_p'].notna()
    if valid_offer.sum() > 0:
        _, pvals_corr, _, _ = multipletests(df_stats.loc[valid_offer, 'offer_p'], method='fdr_bh')
        df_stats.loc[valid_offer, 'offer_p_bh'] = pvals_corr
    df_stats['offer_sig_bh'] = df_stats.get('offer_p_bh', pd.Series(dtype=float)) < 0.05

if 'accept_p' in df_stats.columns:
    valid_acc = df_stats['accept_p'].notna()
    if valid_acc.sum() > 0:
        _, pvals_acc_corr, _, _ = multipletests(df_stats.loc[valid_acc, 'accept_p'], method='fdr_bh')
        df_stats.loc[valid_acc, 'accept_p_bh'] = pvals_acc_corr
    df_stats['accept_sig_bh'] = df_stats.get('accept_p_bh', pd.Series(dtype=float)) < 0.05

for col in ['offer_sig', 'offer_sig_bh', 'accept_sig', 'accept_sig_bh']:
    if col not in df_stats.columns:
        df_stats[col] = False

sig_counts = pd.DataFrame({
    'Metric': ['Offer shift', 'Accept shift'],
    'Sig (raw)': [df_stats['offer_sig'].sum(), df_stats['accept_sig'].sum()],
    'Sig (BH-FDR)': [df_stats['offer_sig_bh'].sum(), df_stats['accept_sig_bh'].sum()],
    'Total configs': [len(df_stats)] * 2
})
print('=== Significance Summary (all paradigms combined) ===')
display(sig_counts)

sig_par = df_stats.groupby('paradigm').agg(
    offer_sig_raw=('offer_sig','sum'),
    offer_sig_bh=('offer_sig_bh','sum'),
    accept_sig_raw=('accept_sig','sum'),
    accept_sig_bh=('accept_sig_bh','sum'),
    total=('dimension','count'),
)
print('\n=== Significance by Paradigm ===')
display(sig_par)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

paradigm_list = sorted(df_stats['paradigm'].unique())
color_map = {p: PALETTE[i] for i, p in enumerate(paradigm_list)}

valid = df_stats.dropna(subset=['offer_cohens_d', 'offer_p'])
colors = valid['paradigm'].map(color_map)
sig_mask = valid['offer_sig_bh'].fillna(False)

axes[0].scatter(valid['offer_cohens_d'], -np.log10(valid['offer_p'].clip(1e-10)),
                c=colors, alpha=0.6, s=35, edgecolors='none')
axes[0].scatter(valid.loc[sig_mask, 'offer_cohens_d'],
                -np.log10(valid.loc[sig_mask, 'offer_p'].clip(1e-10)),
                c=colors[sig_mask], alpha=1.0, s=55, edgecolors='k', linewidths=0.5, marker='*')
axes[0].axhline(-np.log10(0.05), color='red', lw=1, linestyle='--', label='p=0.05')
axes[0].axvline(0, color='grey', lw=0.8, linestyle='--')
axes[0].set_xlabel("Cohen's d (offer shift)")
axes[0].set_ylabel('-log10(p-value)')
axes[0].set_title('(a) Volcano Plot: Offer Shift')
handles = [mpatches.Patch(color=color_map[p], label=p) for p in paradigm_list]
axes[0].legend(handles=handles, fontsize=7)

dim_order = sorted(df_stats['dimension'].dropna().unique())
bp_data = [df_stats.loc[df_stats['dimension']==d, 'offer_cohens_d'].dropna().values for d in dim_order]
bp = axes[1].boxplot(bp_data, labels=dim_order, patch_artist=True, notch=False,
                     medianprops=dict(color='k', lw=1.5))
for patch, color in zip(bp['boxes'], sns.color_palette('husl', len(dim_order))):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1].axhline(0, color='red', lw=0.8, linestyle='--')
axes[1].set_xticklabels(dim_order, rotation=45, ha='right')
axes[1].set_ylabel("Cohen's d")
axes[1].set_title("(b) Effect Size Distribution per Dimension")

plt.suptitle('Statistical Significance Overview (All Paradigms)', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 6 - Top Configurations by Key Metrics

We rank configurations by three criteria:
1. **Accept rate shift** (Delta acceptance rate, pp)
2. **Offer shift** (Delta proposer demand, pp)
3. **Compound payoff shift** (Delta E[payoff], pp)

All metrics are relative to the unsteered baseline.

In [ ]:
def top_table(df, metric_col, n=10, ascending=False):
    cols = ['paradigm','experiment','layer','role','dimension','best_alpha','n_games',
            'bl_accept_rate','fin_accept_rate','delta_accept_rate',
            'bl_mean_proposer_pct','fin_mean_proposer_pct','delta_proposer_pct',
            'bl_compound_payoff','fin_compound_payoff','delta_compound_payoff',
            'offer_cohens_d','offer_p','accept_p',
            'offer_sig_bh','accept_sig_bh']
    available = [c for c in cols if c in df.columns]
    t = df[available].dropna(subset=[metric_col]).sort_values(metric_col, ascending=ascending).head(n)
    return t.round(3)

show_cols = ['paradigm','layer','role','dimension','best_alpha','n_games',
             'bl_accept_rate','fin_accept_rate','delta_accept_rate',
             'offer_cohens_d']
show_cols = [c for c in show_cols if c in df_stats.columns]

print('='*90)
print('TOP 10 CONFIGS: Largest INCREASE in Accept Rate')
print('='*90)
display(top_table(df_stats, 'delta_accept_rate', n=10, ascending=False)[show_cols])

print()
print('-'*90)
print('TOP 10 CONFIGS: Largest DECREASE in Accept Rate')
print('-'*90)
display(top_table(df_stats, 'delta_accept_rate', n=10, ascending=True)[show_cols])

In [ ]:
show_off = ['paradigm','layer','role','dimension','best_alpha','n_games',
            'bl_mean_proposer_pct','fin_mean_proposer_pct','delta_proposer_pct',
            'offer_cohens_d','offer_p','offer_sig_bh']
show_off = [c for c in show_off if c in df_stats.columns]

print('='*90)
print('TOP 10 CONFIGS: Largest INCREASE in Proposer Offer')
print('='*90)
display(top_table(df_stats, 'delta_proposer_pct', n=10, ascending=False)[show_off])

print()
print('-'*90)
print('TOP 10 CONFIGS: Largest DECREASE in Proposer Offer (most generous)')
print('-'*90)
display(top_table(df_stats, 'delta_proposer_pct', n=10, ascending=True)[show_off])

In [ ]:
show_cpd = ['paradigm','layer','role','dimension','best_alpha','n_games',
            'bl_compound_payoff','fin_compound_payoff','delta_compound_payoff',
            'delta_accept_rate','delta_proposer_pct',
            'offer_cohens_d','offer_sig_bh']
show_cpd = [c for c in show_cpd if c in df_stats.columns]

print('='*90)
print('TOP 10 CONFIGS: Best Compound Payoff (E[payoff] = accept_rate x offer_pct)')
print('='*90)
display(top_table(df_stats, 'delta_compound_payoff', n=10, ascending=False)[show_cpd])

print()
print('-'*90)
print('TOP 10 CONFIGS: Worst Compound Payoff (largest decrease)')
print('-'*90)
display(top_table(df_stats, 'delta_compound_payoff', n=10, ascending=True)[show_cpd])

---
## 7 - Heatmaps: Dimension x Layer Effect Landscape

In [ ]:
def make_heatmap(df, metric, title, ax, cmap='RdBu_r', center=0, fmt='.1f'):
    piv = df.pivot_table(values=metric, index='dimension', columns='layer_num', aggfunc='mean')
    if piv.empty:
        ax.set_visible(False)
        return
    vals = piv.values[~np.isnan(piv.values)]
    vmax = max(abs(vals).max(), 0.01) if len(vals) > 0 else 1
    sns.heatmap(piv, ax=ax, cmap=cmap, center=center, vmin=-vmax, vmax=vmax,
                annot=True, fmt=fmt, linewidths=0.5, linecolor='white',
                cbar_kws={'shrink': 0.75, 'label': metric})
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Layer')
    ax.set_ylabel('Dimension')
    ax.tick_params(axis='x', rotation=0)
    ax.tick_params(axis='y', rotation=0)

for par in sorted(df_stats['paradigm'].unique()):
    sub = df_stats[df_stats['paradigm'] == par]
    if sub.empty:
        continue
    for role in sorted(sub['role'].unique()):
        sub_r = sub[sub['role'] == role]
        if sub_r.empty:
            continue
        fig, axes = plt.subplots(1, 3, figsize=(18, max(4, len(sub_r['dimension'].unique())*0.45 + 1)))
        make_heatmap(sub_r, 'delta_accept_rate', 'Delta Accept Rate (pp)', axes[0])
        make_heatmap(sub_r, 'delta_proposer_pct', 'Delta Proposer Offer (%)', axes[1])
        make_heatmap(sub_r, 'delta_compound_payoff', 'Delta Compound Payoff (pp)', axes[2])
        fig.suptitle(f'{par} | Role: {role.capitalize()}', fontsize=13, fontweight='bold')
        plt.tight_layout()
        plt.show()

---
## 8 - Dose-Response: Alpha x Metric Curves

In [ ]:
if not df_games.empty:
    alpha_games = df_games.copy()

    def parse_alpha(tag):
        if tag == 'baseline':
            return 0.0
        if tag == 'final':
            return np.nan
        try:
            return float(str(tag).replace('alpha', ''))
        except:
            return np.nan

    alpha_games['alpha_num'] = alpha_games['alpha_val'].fillna(
        alpha_games['alpha_tag'].map(parse_alpha))
    alpha_games = alpha_games.dropna(subset=['alpha_num'])

    dose = (
        alpha_games
        .groupby(['paradigm','experiment','layer','role','dimension','alpha_num'])
        .agg(
            accept_rate=('agreed','mean'),
            mean_offer=('proposer_pct','mean'),
            n=('agreed','count'),
        )
        .reset_index()
    )
    dose['compound'] = dose['accept_rate'] * dose['mean_offer']

    top_dims = (
        df_stats.groupby('dimension')['offer_cohens_d'].apply(lambda x: x.abs().mean())
        .sort_values(ascending=False).head(6).index.tolist()
    )

    paradigm_list = sorted(dose['paradigm'].unique())
    n_top = len(top_dims)
    fig, axes = plt.subplots(3, n_top, figsize=(4*n_top, 10))
    if n_top == 1:
        axes = axes.reshape(-1, 1)
    metrics    = ['accept_rate', 'mean_offer', 'compound']
    ylabels    = ['Accept Rate', 'Mean Offer (%)', 'Compound Payoff']
    multipliers= [100, 1, 1]

    for col, dim in enumerate(top_dims):
        sub = dose[dose['dimension'] == dim]
        for row, (met, yl, mul) in enumerate(zip(metrics, ylabels, multipliers)):
            ax = axes[row, col]
            for i, par in enumerate(paradigm_list):
                grp = sub[sub['paradigm'] == par]
                if grp.empty:
                    continue
                agg = grp.groupby('alpha_num')[met].mean() * mul
                ax.plot(agg.index, agg.values, 'o-', color=PALETTE[i], label=par,
                        markersize=4, linewidth=1.5)
            ax.axvline(0, color='grey', lw=0.8, linestyle='--')
            if row == 0:
                ax.set_title(dim, fontsize=10, fontweight='bold')
            if col == 0:
                ax.set_ylabel(yl, fontsize=9)
            if row == 2:
                ax.set_xlabel('Alpha')
            if row == 0 and col == 0:
                ax.legend(fontsize=6)

    plt.suptitle("Dose-Response: Steering Intensity vs Outcome Metrics\n(Top 6 Dimensions by Mean |Cohen's d|)",
                 fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print('No game-level data available for dose-response.')

---
## 9 - Role Analysis: Proposer vs Responder Steering

In [ ]:
role_agg = (
    df_stats
    .groupby(['paradigm', 'role'])
    .agg(
        mean_delta_offer     = ('delta_proposer_pct',   'mean'),
        std_delta_offer      = ('delta_proposer_pct',   'std'),
        mean_delta_accept    = ('delta_accept_rate',     'mean'),
        std_delta_accept     = ('delta_accept_rate',     'std'),
        mean_delta_compound  = ('delta_compound_payoff', 'mean'),
        std_delta_compound   = ('delta_compound_payoff', 'std'),
        frac_sig_offer       = ('offer_sig_bh',          'mean'),
        frac_sig_accept      = ('accept_sig_bh',         'mean'),
        n                    = ('dimension',             'count'),
    )
    .round(3)
)
print('=== Role-Level Summary ===')
display(role_agg)

In [ ]:
paradigm_list = sorted(df_stats['paradigm'].unique())
n_par = len(paradigm_list)
fig, axes_arr = plt.subplots(1, 3, figsize=(5*n_par + 2, 5))
metrics = ['delta_proposer_pct', 'delta_accept_rate', 'delta_compound_payoff']
titles  = ['Delta Proposer Offer (%)', 'Delta Accept Rate (pp)', 'Delta Compound Payoff (pp)']
scales  = [1, 100, 1]

for ax, met, title, sc in zip(axes_arr, metrics, titles, scales):
    positions = []
    labels = []
    for i, par in enumerate(paradigm_list):
        grp = df_stats[df_stats['paradigm'] == par]
        roles_present = sorted(grp['role'].unique())
        for j, role in enumerate(roles_present):
            vals = grp.loc[grp['role'] == role, met].dropna() * sc
            if len(vals) == 0:
                continue
            x = i*(len(roles_present)+1) + j
            bp = ax.boxplot(vals, positions=[x], widths=0.6, patch_artist=True,
                            medianprops=dict(color='k', lw=1.5),
                            boxprops=dict(facecolor=PALETTE[i], alpha=0.6))
        positions.append(i*(len(roles_present)+1) + (len(roles_present)-1)/2)
        labels.append(par[:10])
    ax.axhline(0, color='red', lw=0.8, linestyle='--')
    ax.set_xticks(positions)
    ax.set_xticklabels(labels, fontsize=7)
    ax.set_title(title)

plt.suptitle('Role Comparison: Steering Effects by Paradigm', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
for par in sorted(df_stats['paradigm'].unique()):
    sub = df_stats[df_stats['paradigm'] == par]
    piv = sub.pivot_table(values='delta_compound_payoff', index='dimension', columns='role', aggfunc='mean')
    if piv.empty:
        continue
    ax = piv.plot(kind='bar', figsize=(12, 4), edgecolor='k', linewidth=0.5, width=0.7)
    plt.axhline(0, color='k', lw=0.8, linestyle='--')
    plt.title(f'{par}: Compound Payoff Delta by Dimension and Role', fontsize=12, fontweight='bold')
    plt.xlabel('Dimension')
    plt.ylabel('Delta Compound Payoff (pp)')
    plt.xticks(rotation=45, ha='right')
    plt.legend(title='Role')
    plt.tight_layout()
    plt.show()

---
## 10 - Layer Analysis: Where in the Network Does Steering Work?

In [ ]:
layer_agg = (
    df_stats
    .groupby(['paradigm', 'layer_num'])
    .agg(
        mean_abs_cohens_d    = ('offer_cohens_d',       lambda x: x.abs().mean()),
        mean_delta_offer     = ('delta_proposer_pct',   'mean'),
        mean_delta_accept    = ('delta_accept_rate',     'mean'),
        mean_delta_compound  = ('delta_compound_payoff', 'mean'),
        frac_sig             = ('offer_sig_bh',          'mean'),
        n                    = ('dimension',             'count'),
    )
    .reset_index()
    .round(3)
)
print('=== Layer Summary ===')
display(layer_agg)

In [ ]:
fig, axes_arr = plt.subplots(2, 2, figsize=(13, 9))
axes_flat = axes_arr.flatten()

metrics_l = ['mean_abs_cohens_d', 'mean_delta_offer', 'mean_delta_accept', 'mean_delta_compound']
titles_l  = ["Mean |Cohen's d| (offer)", 'Mean Delta Offer (%)', 'Mean Delta Accept Rate', 'Mean Delta Compound Payoff']

paradigm_list = sorted(layer_agg['paradigm'].unique())
for ax, met, title in zip(axes_flat, metrics_l, titles_l):
    for i, par in enumerate(paradigm_list):
        grp = layer_agg[layer_agg['paradigm'] == par]
        grp_s = grp.sort_values('layer_num')
        ax.plot(grp_s['layer_num'], grp_s[met], 'o-', color=PALETTE[i], label=par, linewidth=2, markersize=7)
    ax.axhline(0, color='grey', lw=0.8, linestyle='--')
    ax.set_xlabel('Layer')
    ax.set_ylabel(title)
    ax.set_title(title)
    ax.legend(fontsize=7)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

plt.suptitle('Effect Magnitude vs Network Layer (All Paradigms)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 11 - Dimension Deep-Dives

For each dimension, we show the full distribution of outcomes across configs, and the per-paradigm breakdown.

In [ ]:
dims_all = sorted(df_stats['dimension'].dropna().unique())
n_dims   = len(dims_all)
paradigm_list = sorted(df_stats['paradigm'].unique())
n_par = len(paradigm_list)

fig, axes_arr = plt.subplots(3, max(n_dims, 1), figsize=(2.5*max(n_dims, 1), 9))
if n_dims == 1:
    axes_arr = axes_arr.reshape(-1, 1)

metrics_d = ['delta_proposer_pct', 'delta_accept_rate', 'delta_compound_payoff']
ylabels_d = ['Delta Offer (%)', 'Delta Accept Rate', 'Delta Compound Payoff']

for col, dim in enumerate(dims_all):
    sub = df_stats[df_stats['dimension'] == dim]
    for row, (met, yl) in enumerate(zip(metrics_d, ylabels_d)):
        ax = axes_arr[row, col]
        for i, par in enumerate(paradigm_list):
            grp = sub[sub['paradigm'] == par]
            vals = grp[met].dropna()
            if len(vals) == 0:
                continue
            ax.scatter([i]*len(vals), vals, color=PALETTE[i], alpha=0.7, s=20, zorder=3)
            ax.plot([i-0.3, i+0.3], [vals.mean(), vals.mean()], color=PALETTE[i], lw=2)
        ax.axhline(0, color='grey', lw=0.8, linestyle='--')
        ax.set_xticks(range(n_par))
        ax.set_xticklabels([p[:5] for p in paradigm_list], fontsize=6)
        if col == 0:
            ax.set_ylabel(yl, fontsize=8)
        if row == 0:
            ax.set_title(dim, fontsize=9, fontweight='bold')

plt.suptitle('Per-Dimension Distribution of Effects (All Paradigms)\n(dot = config; bar = mean)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
dim_table = (
    df_stats
    .groupby(['dimension', 'paradigm', 'role'])
    .agg(
        n_configs=('best_alpha','count'),
        mean_delta_offer=('delta_proposer_pct','mean'),
        mean_delta_accept=('delta_accept_rate','mean'),
        mean_delta_compound=('delta_compound_payoff','mean'),
        mean_abs_d=('offer_cohens_d', lambda x: x.abs().mean()),
        frac_sig_offer=('offer_sig_bh','mean'),
        frac_sig_accept=('accept_sig_bh','mean'),
        best_compound_delta=('delta_compound_payoff','max'),
    )
    .round(3)
    .reset_index()
    .sort_values('mean_delta_compound', ascending=False)
)
print('=== Dimension x Paradigm x Role Summary ===')
display(dim_table)

---
## 12 - Cross-Paradigm Comparison

In [ ]:
paradigm_list = sorted(df_stats['paradigm'].unique())

para_aggs = {}
for par in paradigm_list:
    agg = df_stats[df_stats['paradigm'] == par].groupby(['dimension','role']).agg(
        mean_delta_offer=('delta_proposer_pct','mean'),
        mean_delta_accept=('delta_accept_rate','mean'),
        mean_delta_compound=('delta_compound_payoff','mean'),
        mean_abs_d=('offer_cohens_d', lambda x: x.abs().mean()),
    )
    para_aggs[par] = agg

pairs = [(paradigm_list[i], paradigm_list[j])
         for i in range(len(paradigm_list)) for j in range(i+1, len(paradigm_list))]

for p1, p2 in pairs:
    if p1 not in para_aggs or p2 not in para_aggs:
        continue
    a1 = para_aggs[p1].add_suffix(f'_p1')
    a2 = para_aggs[p2].add_suffix(f'_p2')
    cross = a1.join(a2, how='inner').reset_index()
    if cross.empty:
        print(f'No overlapping (dimension, role) between {p1} and {p2}')
        continue

    print(f'\n=== {p1} vs {p2} (matched on dimension x role) ===')
    display(cross.round(3))

    fig, axes_c = plt.subplots(1, 3, figsize=(15, 5))
    met_pairs = [
        ('mean_delta_offer_p1', 'mean_delta_offer_p2', 'Delta Offer (%)'),
        ('mean_delta_accept_p1', 'mean_delta_accept_p2', 'Delta Accept Rate'),
        ('mean_delta_compound_p1', 'mean_delta_compound_p2', 'Delta Compound Payoff'),
    ]
    for ax, (xm, ym, label) in zip(axes_c, met_pairs):
        sub = cross.dropna(subset=[xm, ym])
        if sub.empty:
            ax.set_visible(False)
            continue
        colors_roles = sub['role'].map({'proposer': PALETTE[2], 'responder': PALETTE[3]}).fillna(PALETTE[0])
        ax.scatter(sub[xm], sub[ym], c=colors_roles, s=60, alpha=0.8, edgecolors='k', linewidths=0.4)
        lim = max(sub[[xm, ym]].abs().max().max() * 1.1, 1)
        ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
        ax.plot([-lim, lim], [-lim, lim], 'k--', lw=0.8, label='y=x')
        ax.axhline(0, color='grey', lw=0.6, linestyle=':')
        ax.axvline(0, color='grey', lw=0.6, linestyle=':')
        ax.set_xlabel(f'{p1}: {label}')
        ax.set_ylabel(f'{p2}: {label}')
        ax.set_title(label)
        for _, row in sub.iterrows():
            if abs(row[xm]) > lim*0.4 or abs(row[ym]) > lim*0.4:
                ax.annotate(f"{row['dimension']}\n({row['role'][0]})",
                            (row[xm], row[ym]), fontsize=6, ha='center', va='bottom')
        handles = [mpatches.Patch(color=PALETTE[2], label='proposer'),
                   mpatches.Patch(color=PALETTE[3], label='responder')]
        ax.legend(handles=handles, fontsize=8)

    fig.suptitle(f'{p1} vs {p2}:\nAre Effects Consistent Across Paradigms?',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

    for _, (xm, ym, label) in enumerate(met_pairs):
        sub = cross.dropna(subset=[xm, ym])
        if len(sub) >= 3:
            r, p = stats.pearsonr(sub[xm], sub[ym])
            print(f'  {label}: r={r:.3f}, p={p:.4f}, n={len(sub)}')

---
## 12.5 - LLM-vs-LLM (Clean) vs Teammate Comparison

Direct comparison of matched dimension x layer configs across paradigms, focusing on the LLM-vs-LLM (Clean)
final grid data vs the two teammate paradigms. Includes heatmap view of the Clean effect landscape, scatter of
Cohen's d values across matched configs, and a cross-paradigm summary table.

In [ ]:
paradigm_list = sorted(df_stats['paradigm'].unique())
clean_par = 'LLM-vs-LLM (Clean)'
other_pars = [p for p in paradigm_list if p != clean_par]

if clean_par in paradigm_list and len(other_pars) > 0:
    # --- 12.5a: Heatmap of LLM-vs-LLM (Clean) effect sizes ---
    clean_data = df_stats[df_stats['paradigm'] == clean_par]
    if not clean_data.empty:
        fig, axes_h = plt.subplots(1, 3, figsize=(18, max(4, clean_data['dimension'].nunique()*0.45 + 1)))
        make_heatmap(clean_data, 'offer_cohens_d', "Cohen's d (offer shift)", axes_h[0], fmt='.2f')
        make_heatmap(clean_data, 'delta_proposer_pct', 'Delta Offer (%)', axes_h[1])
        make_heatmap(clean_data, 'delta_compound_payoff', 'Delta Compound Payoff (pp)', axes_h[2])
        fig.suptitle('LLM-vs-LLM (Clean): Effect Landscape (Dimension x Layer)', fontsize=13, fontweight='bold')
        plt.tight_layout()
        plt.show()

    # --- 12.5b: Scatter of Cohen's d: Clean vs each other paradigm ---
    for other in other_pars:
        clean_agg = (df_stats[df_stats['paradigm'] == clean_par]
                     .groupby(['dimension', 'layer_num'])['offer_cohens_d']
                     .mean().reset_index().rename(columns={'offer_cohens_d': 'd_clean'}))
        other_agg = (df_stats[df_stats['paradigm'] == other]
                     .groupby(['dimension', 'layer_num'])['offer_cohens_d']
                     .mean().reset_index().rename(columns={'offer_cohens_d': 'd_other'}))
        merged = pd.merge(clean_agg, other_agg, on=['dimension', 'layer_num'], how='inner')

        if merged.empty:
            print(f'No overlapping dimension x layer between {clean_par} and {other}')
            continue

        fig, ax = plt.subplots(1, 1, figsize=(7, 7))
        ax.scatter(merged['d_clean'], merged['d_other'], s=60, alpha=0.7,
                   edgecolors='k', linewidths=0.4, c=PALETTE[0])
        lim = max(merged[['d_clean','d_other']].abs().max().max() * 1.15, 0.5)
        ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
        ax.plot([-lim, lim], [-lim, lim], 'k--', lw=0.8, label='y=x')
        ax.axhline(0, color='grey', lw=0.6, linestyle=':')
        ax.axvline(0, color='grey', lw=0.6, linestyle=':')
        ax.set_xlabel(f"{clean_par} Cohen's d")
        ax.set_ylabel(f"{other} Cohen's d")
        ax.set_title(f"Cohen's d: {clean_par} vs {other}")
        for _, row in merged.iterrows():
            ax.annotate(f"{row['dimension']}\nL{row['layer_num']}", (row['d_clean'], row['d_other']),
                        fontsize=6, ha='center', va='bottom', alpha=0.7)
        ax.legend()
        plt.tight_layout()
        plt.show()

        if len(merged) >= 3:
            r, p = stats.pearsonr(merged['d_clean'], merged['d_other'])
            print(f"  Correlation: r={r:.3f}, p={p:.4f}, n={len(merged)}")

    # --- 12.5c: Summary comparison table ---
    summary_cross = (
        df_stats
        .groupby('paradigm')
        .agg(
            n_configs=('dimension', 'count'),
            mean_abs_d=('offer_cohens_d', lambda x: x.abs().mean()),
            mean_delta_offer=('delta_proposer_pct', 'mean'),
            mean_delta_accept=('delta_accept_rate', 'mean'),
            mean_delta_compound=('delta_compound_payoff', 'mean'),
            frac_sig=('offer_sig_bh', 'mean'),
            median_n=('n_games', 'median'),
        )
        .round(3)
    )
    print('\n=== Cross-Paradigm Summary ===')
    display(summary_cross)

elif clean_par not in paradigm_list:
    print('LLM-vs-LLM (Clean) data not available.')
else:
    print('No teammate data available for comparison.')

---
## 13 - Full Statistical Summary Tables

In [ ]:
pub_cols = ['dimension', 'role', 'layer_num', 'best_alpha', 'n_games',
            'bl_accept_rate', 'fin_accept_rate', 'delta_accept_rate',
            'bl_mean_proposer_pct', 'fin_mean_proposer_pct', 'delta_proposer_pct',
            'bl_compound_payoff', 'fin_compound_payoff', 'delta_compound_payoff',
            'offer_cohens_d', 'offer_p', 'offer_sig_bh',
            'accept_p', 'accept_sig_bh']
pub_rename = {
    'dimension': 'Dimension', 'role': 'Role', 'layer_num': 'Layer', 'best_alpha': 'Best alpha',
    'n_games': 'n',
    'bl_accept_rate': 'Acc (BL)', 'fin_accept_rate': 'Acc (S)', 'delta_accept_rate': 'Delta Acc',
    'bl_mean_proposer_pct': 'Offer (BL)', 'fin_mean_proposer_pct': 'Offer (S)', 'delta_proposer_pct': 'Delta Offer',
    'bl_compound_payoff': 'Cpd (BL)', 'fin_compound_payoff': 'Cpd (S)', 'delta_compound_payoff': 'Delta Cpd',
    'offer_cohens_d': "d (offer)", 'offer_p': 'p (offer)', 'offer_sig_bh': 'Sig* (offer)',
    'accept_p': 'p (accept)', 'accept_sig_bh': 'Sig* (accept)'
}

for par in sorted(df_stats['paradigm'].unique()):
    sub = df_stats[df_stats['paradigm'] == par].copy()
    available = [c for c in pub_cols if c in sub.columns]
    sub = sub[available].rename(columns=pub_rename)
    sort_cols = [c for c in ['Dimension', 'Role', 'Layer'] if c in sub.columns]
    sub = sub.sort_values(sort_cols)
    print(f'\n=== {par}: Full Results Table ===')
    display(sub.round(3))

---
## 14 - Acceptance Curve Analysis

We visualise how acceptance rate varies as a function of the *proposer's offer* for baseline vs steered games.
A non-monotonic acceptance curve indicates RLHF fairness enforcement (rejecting overly greedy AND overly generous offers).

In [ ]:
if not df_games.empty:
    df_games['offer_bin'] = pd.cut(df_games['proposer_pct'], bins=np.arange(0, 101, 10),
                                   include_lowest=True)
    df_games['offer_bin_mid'] = df_games['offer_bin'].apply(
        lambda x: x.mid if not pd.isna(x) else np.nan)

    paradigm_list = sorted(df_games['paradigm'].unique())
    n_par = len(paradigm_list)
    fig, axes_acc = plt.subplots(1, n_par, figsize=(5*n_par + 2, 5), sharey=True)
    if n_par == 1:
        axes_acc = [axes_acc]

    for ax_idx, par in enumerate(paradigm_list):
        sub = df_games[df_games['paradigm'] == par]
        ax  = axes_acc[ax_idx]
        for tag, grp in sub.groupby('alpha_tag'):
            if tag == 'final':
                continue
            agg = grp.groupby('offer_bin_mid')['agreed'].agg(['mean', 'count']).reset_index()
            agg = agg[agg['count'] >= 5]
            is_bl = tag == 'baseline'
            lw = 2.5 if is_bl else 1.2
            ls = '-'  if is_bl else '--'
            col = 'black' if is_bl else None
            label = 'baseline' if is_bl else tag
            ax.plot(agg['offer_bin_mid'], agg['mean']*100, lw=lw, ls=ls,
                    color=col, label=label, alpha=0.8 if not is_bl else 1.0)

        ax.axvline(50, color='red', lw=1, linestyle=':', label='50% split')
        ax.set_xlabel('Proposer offer (% of pool)')
        if ax_idx == 0:
            ax.set_ylabel('Acceptance rate (%)')
        ax.set_title(par)
        ax.set_ylim(-5, 110)
        ax.legend(fontsize=6, ncol=2)

    plt.suptitle('Acceptance Curve: Rate vs Offer Level (Baseline vs All Alphas)',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('No game data available for acceptance curve.')

---
## 15 - Parse Error and Data Quality Audit

In [ ]:
if not df_games.empty and 'parse_error' in df_games.columns:
    parse_err = (
        df_games
        .groupby(['paradigm', 'alpha_tag'])
        .agg(n=('parse_error','count'), errors=('parse_error','sum'))
        .assign(error_rate=lambda x: x['errors'] / x['n'])
        .reset_index()
        .sort_values('error_rate', ascending=False)
    )
    print('=== Parse Error Rates ===')
    errs_only = parse_err[parse_err['errors'] > 0]
    if not errs_only.empty:
        display(errs_only)
    else:
        print('No parse errors detected in any paradigm.')

    total_errors = parse_err['errors'].sum()
    total_games  = parse_err['n'].sum()
    print(f'\nOverall parse error rate: {total_errors}/{total_games} = {total_errors/total_games:.3%}')
else:
    print('No parse error data available.')

---
## 16 - Summary: Key Findings

In [ ]:
print('='*80)
print('               SUMMARY: KEY EMPIRICAL FINDINGS')
print('='*80)

for par in sorted(df_stats['paradigm'].unique()):
    sub = df_stats[df_stats['paradigm'] == par]
    if sub.empty:
        continue
    n_total = len(sub)
    n_sig_offer  = sub['offer_sig_bh'].sum()
    n_sig_accept = sub['accept_sig_bh'].sum()

    valid_cpd = sub.dropna(subset=['delta_compound_payoff'])
    if valid_cpd.empty:
        continue

    best_cpd = valid_cpd.loc[valid_cpd['delta_compound_payoff'].idxmax()]
    worst_cpd= valid_cpd.loc[valid_cpd['delta_compound_payoff'].idxmin()]

    print(f'\n-- {par} --')
    print(f'  Configurations:   {n_total}')
    if 'n_games' in sub.columns:
        print(f'  Median n/config:  {sub["n_games"].median():.0f}')
    print(f'  Sig offer shift (BH): {n_sig_offer}/{n_total} = {n_sig_offer/n_total:.1%}')
    print(f'  Sig accept shift (BH): {n_sig_accept}/{n_total} = {n_sig_accept/n_total:.1%}')
    print(f'  Mean Delta offer:    {sub["delta_proposer_pct"].mean():+.2f}%')
    print(f'  Mean Delta accept:   {sub["delta_accept_rate"].mean()*100:+.2f}pp')
    print(f'  Mean Delta compound: {sub["delta_compound_payoff"].mean():+.2f}pp')
    print(f'  Best config:  {best_cpd["dimension"]:15s} | L{best_cpd["layer_num"]} a={best_cpd["best_alpha"]} -> Delta={best_cpd["delta_compound_payoff"]:+.1f}pp')
    print(f'  Worst config: {worst_cpd["dimension"]:15s} | L{worst_cpd["layer_num"]} a={worst_cpd["best_alpha"]} -> Delta={worst_cpd["delta_compound_payoff"]:+.1f}pp')

print()
print('  NOTE: Compound payoff = accept_rate x offer_pct.')
print('  Positive Delta compound = steering improves expected earnings.')
print('  BH-FDR correction applied across all configs.')

In [ ]:
dim_means = (
    df_stats
    .groupby(['paradigm', 'dimension'])
    ['delta_compound_payoff']
    .mean()
    .unstack('paradigm')
)

paradigm_list = sorted(dim_means.columns)
dims_r = dim_means.index.tolist()
N = len(dims_r)
if N > 0:
    angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

    for i, par in enumerate(paradigm_list):
        vals = dim_means[par].fillna(0).tolist()
        vals += vals[:1]
        ax.plot(angles, vals, 'o-', color=PALETTE[i], linewidth=2, label=par)
        ax.fill(angles, vals, color=PALETTE[i], alpha=0.1)

    ax.set_thetagrids(np.degrees(angles[:-1]), dims_r, fontsize=8)
    ax.set_title('Compound Payoff Delta by Dimension\n(mean across layers and roles)',
                 fontsize=11, fontweight='bold', pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.45, 1.1), fontsize=8)
    ax.axhline(0, color='grey', lw=0.8, linestyle='--')
    plt.tight_layout()
    plt.show()
else:
    print('No dimension data for radar chart.')

---
## Appendix A - Full Per-Alpha Detail Table

In [ ]:
if not df_games.empty:
    alpha_games_app = df_games.copy()
    alpha_games_app['alpha_num'] = alpha_games_app['alpha_val']
    if 'alpha_tag' in alpha_games_app.columns:
        def _parse_alpha_app(tag):
            if tag == 'baseline':
                return 0.0
            if tag == 'final':
                return np.nan
            try:
                return float(str(tag).replace('alpha',''))
            except:
                return np.nan
        mask = alpha_games_app['alpha_num'].isna()
        alpha_games_app.loc[mask, 'alpha_num'] = alpha_games_app.loc[mask, 'alpha_tag'].apply(_parse_alpha_app)
    alpha_games_app = alpha_games_app.dropna(subset=['alpha_num'])

    per_alpha = (
        alpha_games_app
        .groupby(['paradigm','experiment','layer','role','dimension','alpha_num'])
        .agg(
            n=('agreed','count'),
            accept_rate=('agreed','mean'),
            mean_offer=('proposer_pct','mean'),
            std_offer=('proposer_pct','std'),
        )
        .round(3)
        .reset_index()
        .sort_values(['paradigm','dimension','role','layer','alpha_num'])
    )
    per_alpha['compound'] = (per_alpha['accept_rate'] * per_alpha['mean_offer']).round(3)
    print('=== Per-Alpha Aggregated Results (Appendix) ===')
    display(per_alpha)
else:
    print('No game data for per-alpha table.')

---
## Appendix B - Experiment-Level Metadata

In [ ]:
exp_meta = (
    df_stats
    .groupby(['paradigm', 'experiment'])
    .agg(
        layers=('layer_num', lambda x: sorted(x.unique())),
        roles=('role', lambda x: sorted(x.unique())),
        dimensions=('dimension', lambda x: sorted(x.unique())),
        n_configs=('dimension','count'),
        mean_delta_compound=('delta_compound_payoff','mean'),
    )
    .reset_index()
)
print('=== Experiment Metadata ===')
display(exp_meta)